In [ ]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

n_data = 1_000_000   
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

query = np.random.random((1, dim)).astype(np.float32)
k = 10

print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start    

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== Annoy ===
Build time: 6.51157546043396 detik
Query time: 0.00017786026000976562 detik
Neighbors: [465912, 768961, 902092, 189155, 497659] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.026200056076049805 detik
Query time: 0.06150031089782715 detik
Neighbors: [465912 768961 902092 189155 497659] ...

=== HNSW (hnswlib) ===
Build time: 37.47144150733948 detik
Query time: 0.041053056716918945 detik
Neighbors: [465912 768961 902092 189155 497659] ...
